# บทที่ 1 — เปิดกล้อง และเข้าใจว่าภาพจากกล้องคืออะไร

> ก่อนเข้าบทนี้ควรทำบทที่ 0 มาก่อน (รู้จัก numpy array, `[y, x]`, shape) ถ้ายังไม่ได้ทำ ย้อนกลับไปทำก่อนนะ

🎯 เป้าหมาย: เปิดกล้อง ถ่าย 1 ภาพ แล้วพิสูจน์ว่ามันคือ numpy array จริงๆ ตามที่บทที่แล้วสอน + รู้จัก "กับดัก" อันโด่งดังของ OpenCV เรื่องสี

## 1. เปิดกล้องด้วย OpenCV

`cv2.VideoCapture(index)` คือคำสั่งเปิดกล้อง — `index` คือ "กล้องตัวที่เท่าไร" ในเครื่อง
(0 = กล้องตัวแรกที่เจอ ปกติคือกล้องในตัวโน้ตบุ๊ก, 1, 2, ... = กล้องอื่นๆ ที่ต่อเพิ่ม เช่นมือถือผ่าน DroidCam)

**หมายเหตุสำคัญ**: ไฟล์นี้ถูกออกแบบให้ **รันได้แม้ไม่มีกล้อง** — ถ้าหาไม่เจอ จะใช้ "ภาพจำลองสนาม" แทนอัตโนมัติ
เพื่อให้ทุกเซลล์ถัดไปยังรันต่อได้ปกติ (ถ้ามีกล้องจริง โค้ดจะใช้ภาพจากกล้องจริงให้เองโดยอัตโนมัติ)

In [ ]:
import cv2
import numpy as np

def open_camera_or_placeholder(index=0):
    # พยายามถ่ายจากกล้องจริงก่อน ถ้าไม่มีกล้องจะใช้ "ภาพจำลองสนาม" แทน
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW)   # CAP_DSHOW = ให้เปิดเร็วขึ้นบน Windows
    if cap.isOpened():
        ok, frame = cap.read()      # .read() คืนค่า 2 ตัว: ok (สำเร็จไหม) และ frame (ภาพ)
        cap.release()               # ปิดกล้องทันทีหลังถ่ายเสร็จ (สำคัญ! ไม่งั้นแอปอื่นใช้กล้องไม่ได้)
        if ok:
            print(f"✅ ถ่ายจากกล้องจริงสำเร็จ! (index {index})")
            return frame
    print("⚠️ ไม่พบกล้อง (ปกติถ้าเครื่องนี้ไม่มีกล้อง หรือกล้องถูกแอปอื่นใช้อยู่)")
    print("   ใช้ 'ภาพจำลองสนาม' แทนไปก่อน — มีกล้องจริงเมื่อไรลองรันเซลล์นี้ใหม่ได้เลย")
    return _placeholder_scene()


def _placeholder_scene():
    # สร้างภาพจำลอง "สนามยิง" มีตุ๊กตา 3 ตัวเหมือนโปรเจคจริง ไว้ใช้ตอนไม่มีกล้อง
    img = np.full((480, 640, 3), (235, 230, 220), np.uint8)          # พื้นหลังห้อง
    cv2.rectangle(img, (0, 340), (640, 480), (170, 140, 90), -1)     # โต๊ะวางตุ๊กตา
    # (x, y, รัศมี, สี BGR) — ยิ่งไกลยิ่งเล็ก เหมือนของจริง
    dolls = [(160, 300, 55, (40, 170, 40)),    # เขียว  = ไดโนเสาร์ (ใกล้สุด ใหญ่สุด)
             (330, 290, 40, (60, 110, 150)),   # น้ำตาล = คาปิบาร่า (ระยะกลาง)
             (480, 280, 28, (130, 130, 130))]  # เทา    = ช้าง (ไกลสุด เล็กสุด)
    for x, y, r, color in dolls:
        cv2.circle(img, (x, y), r, color, -1)
    return img


frame = open_camera_or_placeholder()
print("ชนิดตัวแปร:", type(frame))
print("shape:", frame.shape)
print("dtype:", frame.dtype)   # uint8 = จำนวนเต็ม 0-255 (8 บิต) — มาตรฐานของภาพดิจิทัลทั่วไป

สังเกตว่า `frame` คือ `numpy.ndarray` เป๊ะตามที่บทที่ 0 บอกไว้! `shape` จะเป็น `(สูง, กว้าง, 3)`
— ภาพจากกล้องก็คือ "ตารางตัวเลข 3 มิติ" เหมือนตัวอย่างในบทที่แล้วทุกประการ ต่างกันแค่ขนาดใหญ่กว่ามาก (หลักแสนตัวเลข!)

## 2. ดูภาพด้วย matplotlib (ทำไมไม่ใช้ cv2.imshow ในสมุดบันทึกนี้?)

ปกติสคริปต์ธรรมดาใช้ `cv2.imshow()` เปิดหน้าต่างแยกโชว์ภาพ (ดูตัวอย่างได้ที่ `learn/01_camera.py`)
แต่ใน **สมุดบันทึก (notebook)** เราจะใช้ `plt.imshow()` แทน เพราะ:
- ภาพจะโผล่ **ในสมุดบันทึกเลย** ไม่ต้องสลับหน้าต่าง เหมาะกับการเรียนทีละสเต็ป
- ไม่มีปัญหาหน้าต่างค้าง/kernel แฮงก์ที่บางเครื่องเจอกับ cv2.imshow ในสมุดบันทึก

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Leelawadee UI"   # ฟอนต์นี้รองรับภาษาไทย กันตัวอักษรไทยในกราฟกลายเป็นกล่องว่าง

plt.imshow(frame)
plt.title("ลองแสดงภาพดิบๆ ตรงๆ เลย")
plt.show()

### 🐛 เจอกับดักแล้ว! สีเพี้ยน (โทนฟ้า/ส้มดูแปลกๆ)

นี่ไม่ใช่บั๊ก — **OpenCV เก็บภาพเป็นลำดับสี "B, G, R" (น้ำเงิน-เขียว-แดง)** แต่ `matplotlib` คาดหวังลำดับ "R, G, B"
ผลคือช่องสีแดงกับน้ำเงินถูกสลับที่กัน ภาพเลยดูสีเพี้ยน ต้องสลับกลับด้วย `cv2.cvtColor(...)` ก่อนเสมอเวลาจะโชว์ด้วย matplotlib

In [ ]:
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)   # แปลงลำดับสี BGR -> RGB

plt.imshow(frame_rgb)
plt.title("หลังแปลง BGR -> RGB (สีถูกต้องแล้ว)")
plt.show()

**⚠️ จำกฎนี้ไว้ตลอดทั้งโปรเจค**: ทุกครั้งที่จะโชว์ภาพด้วย matplotlib ต้อง `cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)` ก่อน
แต่ถ้าจะ**ประมวลผล** ภาพ (หาสี, หาวัตถุ) ปล่อยเป็น BGR แบบเดิมได้เลย ฟังก์ชัน OpenCV ส่วนใหญ่ (เช่น `cv2.inRange` ที่จะเจอบทหน้า) ก็ทำงานบน BGR/HSV ไม่ใช่ RGB

## 3. อ่านค่าสีของ pixel เฉพาะจุด (ฝึก indexing ต่อจากบทที่ 0)

In [ ]:
h, w = frame.shape[:2]     # เอาแค่ 2 ค่าแรกของ shape (สูง, กว้าง) ไม่เอาค่าที่ 3 (จำนวนช่องสี)
cy, cx = h // 2, w // 2    # // คือหารแล้วปัดเศษทิ้ง (integer division) หาพิกัดกึ่งกลางภาพ

b, g, r = frame[cy, cx]    # ตามกฎบทที่ 0: [y, x] แถวมาก่อน! ได้ค่าออกมาเป็น B, G, R
print(f"พิกเซลกึ่งกลางภาพ ({cx}, {cy}) มีค่า B={b} G={g} R={r}")

**🧪 ลองเอง:** พิมพ์ค่าสีของพิกเซลที่มุมบนซ้ายสุดของภาพ (แถว 0, คอลัมน์ 0)

<details><summary>👉 คลิกดูเฉลย</summary>

```python
print(frame[0, 0])
```
</details>

In [ ]:
# เขียนคำตอบตรงนี้

## 4. Crop — ตัดเอาเฉพาะส่วนที่สนใจ

ใช้ slicing แบบที่เรียนในบทที่ 0 เป๊ะๆ `frame[y1:y2, x1:x2]`

In [ ]:
crop = frame[100:300, 200:450]   # ตัดเอาแถว 100-300, คอลัมน์ 200-450

print("ขนาดภาพเดิม:", frame.shape)
print("ขนาดหลัง crop:", crop.shape)

plt.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
plt.title("ภาพที่ครอบตัดมา (crop)")
plt.show()

## 5. วาดทับลงบนภาพ (สี่เหลี่ยม / เส้น / ข้อความ)

การ "วาดกรอบรอบวัตถุที่เจอ" ที่จะทำบทหน้า ก็ใช้หลักการเดียวกับตรงนี้เป๊ะ

In [ ]:
drawn_frame = frame.copy()   # .copy() สำคัญมาก! ถ้าไม่ copy จะไปแก้ภาพต้นฉบับโดยไม่ตั้งใจ

cv2.rectangle(drawn_frame, (200, 100), (450, 300), (0, 255, 0), 3)
#              ภาพ            มุมบนซ้าย    มุมล่างขวา   สี(B,G,R)  ความหนาเส้น(px)

cv2.putText(drawn_frame, "ตรงนี้ไง!", (200, 90),
            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
#            ภาพ            ข้อความ      ตำแหน่ง     ฟอนต์                ขนาด  สี         ความหนา

plt.imshow(cv2.cvtColor(drawn_frame, cv2.COLOR_BGR2RGB))
plt.title("วาดกรอบ + ข้อความทับภาพ")
plt.show()

## 🧪 แบบฝึกหัดท้ายบท

โจทย์: เขียนโค้ดที่
1. Crop เอาเฉพาะ **ครึ่งซ้าย** ของภาพ (คอลัมน์ 0 ถึงกึ่งกลาง)
2. วาดวงกลมสีแดง (`cv2.circle`) รัศมี 30px ตรงกึ่งกลางของภาพที่ crop มา
3. แสดงผลด้วย matplotlib

ใบ้: `cv2.circle(ภาพ, (x, y), รัศมี, สี, ความหนา)` — ลองเปิดเอกสาร cv2.circle หรือเดาจากรูปแบบ cv2.rectangle ด้านบนดู

In [ ]:
# เขียนคำตอบตรงนี้

<details><summary>👉 คลิกดูเฉลย (ลองเองก่อนนะ!)</summary>

```python
h, w = frame.shape[:2]
left_half = frame[:, :w // 2].copy()

hh, hw = left_half.shape[:2]
cv2.circle(left_half, (hw // 2, hh // 2), 30, (0, 0, 255), 3)

plt.imshow(cv2.cvtColor(left_half, cv2.COLOR_BGR2RGB))
plt.title("left_half + วงกลมแดงกึ่งกลาง")
plt.show()
```
</details>

---
## ✅ สรุปบทนี้

| เรื่อง | สรุปสั้นๆ |
|---|---|
| `cv2.VideoCapture(index)` | เปิดกล้อง, `.read()` ถ่าย 1 เฟรม |
| BGR vs RGB | OpenCV=BGR, matplotlib=RGB ต้อง `cvtColor` ก่อนโชว์เสมอ |
| `frame[y1:y2, x1:x2]` | crop ภาพ |
| `cv2.rectangle / circle / putText` | วาดทับภาพ |
| `.copy()` | ก่อนวาดทับ ป้องกันแก้ภาพต้นฉบับ |

➡️ **ไปต่อ:** เปิด `02_color_and_hsv.ipynb` เพื่อเรียนรู้วิธีให้คอมพิวเตอร์ "แยกสี" หาตุ๊กตาแต่ละตัว

🎥 **อยากเห็นวิดีโอสด (ไม่ใช่ภาพนิ่ง)?** ลองรัน `learn/01_camera.py` (สคริปต์ธรรมดา ไม่ใช่สมุดบันทึก)